# Tiling: scenes -> 512x512 authentic patches (issue #28)

Reads the preprocessed scenes on Google Drive, cuts them into 512x512 patches,
assigns each scene to train / val / test, and puts the result on Drive as one zip.
Runs on a CPU runtime (no GPU needed).

| Cell | What it does |
|---|---|
| 1 | Parameters (your values) |
| 2 | Mount Drive, get the code, install the packages |
| 3 | List the scenes found on Drive, with their orbit |
| 4 | Tile the scenes on the VM's disk (balancing asc / desc) |
| 5 | Look at a few patches |
| 6 | Zip the patches onto Drive |

In [ ]:
REPO_URL = "https://github.com/bonlieuguillaume/aletheia.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/aletheia"

SCENES_DIR = DRIVE_ROOT + "/data/scenes"          # one sub-folder per scene: .tif or BEAM-DIMAP + .xml,
                                                  # with "asc" or "desc" in the folder name
PATCHES_ZIP = DRIVE_ROOT + "/data/patches_v0.1.zip"
LOCAL_DIR = "/content/patches"                    # VM disk, fast

PATCHES_PER_SCENE = 200
MAX_NODATA_FRACTION = 0.05     # reject a patch with more than 5 % no-data pixels
SPLIT_COUNTS = {"val": 2, "test": 2}
FORCED_SPLITS = {}             # e.g. {"scene_name": "test"} to choose some scenes by hand
ORBITS = {}                    # e.g. {"scene_name": "asc"}; scenes not listed: read from the name
BALANCE_ORBITS = True          # mirror east-west some scenes of the majority orbit to reach 50/50
SEED = 0

In [ ]:
# Mount Google Drive, get the latest code from GitHub, install the packages.
# Everything happens on the Colab VM: nothing is installed on your PC.
import os
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/aletheia"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run(["pip", "install", "-q", "-r", REPO_DIR + "/requirements.txt"], check=True)

# The notebook runs from its own folder and imports the module next to it;
# the repository root on sys.path makes `from features.<name>...` work too.
os.chdir(REPO_DIR + "/features/tiling")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
from tiling import find_scenes, scene_orbits

scenes = find_scenes(SCENES_DIR)
orbits = scene_orbits(list(scenes), ORBITS)   # stops here if an orbit is unknown
for name, path in scenes.items():
    # For a BEAM-DIMAP band the pixels are in the .img next to the .hdr.
    data_file = path.with_suffix(".img") if path.suffix == ".hdr" else path
    print(f"{orbits[name]:4s}  {name}: {path.name} ({data_file.stat().st_size / 1e9:.2f} GB)")

In [ ]:
from tiling import tile_scenes

patches = tile_scenes(SCENES_DIR, LOCAL_DIR, patches_per_scene=PATCHES_PER_SCENE,
                      max_nodata_fraction=MAX_NODATA_FRACTION, split_counts=SPLIT_COUNTS,
                      forced_splits=FORCED_SPLITS, orbits=ORBITS, balance_orbits=BALANCE_ORBITS,
                      seed=SEED)
patches.groupby(["split", "geometry", "scene"]).size()

In [ ]:
import matplotlib.pyplot as plt

from features.evaluation.evaluation import to_display
from features.sar_io.sar_io import read_tif

examples = patches.sample(4, random_state=SEED)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (_, row) in zip(axes, examples.iterrows()):
    ax.imshow(to_display(read_tif(f"{LOCAL_DIR}/real/{row['patch_id']}.tif")), cmap="gray")
    ax.set_title(f"{row['patch_id']} ({row['scene']}, {row['split']})")
    ax.axis("off")
plt.show()

In [ ]:
from features.sar_io.sar_io import zip_folder

zip_folder(LOCAL_DIR, PATCHES_ZIP)